## tl;dr

The 2026-08-06 active512 measurement is genuinely low in its native 8-bit representation: the median is 2 DN, 52.6% of samples are at or below 2 DN, and 7.95% are zero. This is a secondary SNR/quantization limitation, but it is not supported as the primary cause of poor recovery: the active512 measurement mean increased about 4x from 2026-08-05 to 2026-08-06 while mean PBR changed from 14.20 to 13.64, and within the latest run measurement intensity has essentially zero correlation with PBR (Pearson r=0.008).

## Context & Methods

This diagnostic scans the headerless uint16 memmap at grain `(probe frame, camera y, camera x)`, profiles all 1,073,741,824 samples, joins per-output-pixel measurement means to the 16,384-point pixel-wise focus result, and compares preserved 2026-08-04/05/06 quality summaries.

### Key Assumptions

- `Polarized8` is the sensor precision. Scaling samples into 0-4095 uint16 storage does not add precision.
- Peak distance from the requested target is the localization-quality measure; target PBR alone can look acceptable even when the brightest peak is elsewhere.
- Cross-run comparisons involving the legacy full-field mapping are contextual, not controlled causal tests.

In [ ]:
from pathlib import Path
from types import SimpleNamespace
import json
import matplotlib.pyplot as plt
from diagnose_measurement_intensity_128 import build_diagnostic

ROOT = Path.cwd()
OUTPUT = ROOT / 'measurement_intensity_diagnostic_20260806.json'

## Data

The source paths below bind this notebook to the measurement, reconstruction metadata, and focus reports used for the diagnosis.

In [ ]:
args = SimpleNamespace(
    measurement='measurements_128_px4_active512_full_memmap.npy',
    current_quality='measurements_128_px4_active512_full_quality_summary_20260806.json',
    previous_active_quality='measurements_128_px4_active512_full_quality_summary_20260805.json',
    legacy_quality='measurements_128_px4_full_quality_summary.json',
    current_focus_summary='pixelwise_focus_results_128_px4_active512/pixelwise_focus_128_px4_active512_20260806_161522_stride1_batch1000_n16384_summary.json',
    current_focus_points='pixelwise_focus_results_128_px4_active512/pixelwise_focus_128_px4_active512_20260806_161522_stride1_batch1000_n16384_points.csv',
    previous_active_focus_summary='pixelwise_focus_results_128_px4_active512/pixelwise_focus_128_px4_active512_20260805_211916_stride1_n16384_summary.json',
    legacy_focus_summary='pixelwise_focus_results_128_px4/pixelwise_focus_128_px4_20260804_124454_stride1_n16384_summary.json',
    current_recovery='tm_reconstruction_128_px4_active512.json',
    legacy_recovery='tm_reconstruction_128_px4.json',
    output=str(OUTPUT),
)
diagnostic = build_diagnostic(args)
print(f"Saved {OUTPUT}")

## Results

The first visual shows the native 8-bit intensity distribution. The second verifies that batch-level mean intensity is stable enough that temporal drift is not a leading explanation.

In [ ]:
bins = diagnostic['native_histogram_bins']
batch = diagnostic['batch_profile']
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar([row['native_code_bin'] for row in bins], [100*row['sample_fraction'] for row in bins], color='#2F6BFF')
axes[0].set(title='Native 8-bit intensity distribution', xlabel='Native code bin', ylabel='Share of samples (%)')
axes[0].tick_params(axis='x', rotation=35)
axes[1].plot([row['batch'] for row in batch], [row['mean_intensity'] for row in batch], color='#2F6BFF')
axes[1].set(title='Batch mean intensity', xlabel='Batch', ylabel='Stored intensity (0-4095)')
for axis in axes: axis.grid(alpha=0.2)
fig.tight_layout()
plt.show()

In [ ]:
profile = diagnostic['current_profile']
focus = diagnostic['focus_diagnostics']
checks = {
    'native_median_dn': profile['native_8bit_quantiles']['0.5'],
    'fraction_le_2_dn': profile['at_or_below_2_native_codes_fraction'],
    'zero_fraction': profile['zero_fraction'],
    'measurement_mean_vs_pbr_r': focus['measurement_mean_vs_pbr_pearson'],
    'within_2px_fraction': focus['within_2px_fraction'],
    'over_20px_fraction': focus['over_20px_fraction'],
}
print(json.dumps(checks, indent=2))
assert profile['native_8bit_quantiles']['0.5'] == 2
assert abs(focus['measurement_mean_vs_pbr_pearson']) < 0.05
assert len(diagnostic['run_comparison']) == 3

## Takeaways

- Low native sensor occupancy is verified and should be improved because it makes quantization and dark/read noise more important.
- It is not the primary explanation for the current recovery pattern: a 4x active512 intensity change did not improve PBR, and per-output intensity does not predict PBR or localization failure.
- The latest result is spatially mixed: 55.7% of targets place the brightest peak within 2 px, while 33.9% miss by more than 20 px. That pattern is more consistent with mapping/registration/encoding failures than with one uniform SNR limit.
- The highest-value next test is a controlled small-pattern exposure sweep plus a coordinate impulse test, with a real dark frame and native higher-bit acquisition if the camera supports it.